In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone the approved repository branch and record the commit
import os, subprocess
REPO_URL = 'https://github.com/YasinEkici/hyperkvasir-multi-cnn-fusion.git'
BRANCH = 'week3.5/perf-colab'
REPO_DIR = '/content/hyperkvasir-multi-cnn-fusion'
# Public repo: no token needed — just run. For a private repo, set
# os.environ['GITHUB_TOKEN'] = '...' in a scratch cell BEFORE running this one.
token = os.environ.get('GITHUB_TOKEN', '')
clone_url = REPO_URL.replace('https://', f'https://x-access-token:{token}@') if token else REPO_URL
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', 'checkout', BRANCH], check=True)
    subprocess.run(['git', 'pull', '--ff-only'], check=True)
elif os.path.exists(REPO_DIR):
    raise RuntimeError(f'Path exists but is not a git repo: {REPO_DIR}. Restart the runtime.')
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', clone_url, REPO_DIR], check=True)
os.chdir(REPO_DIR)
!git rev-parse HEAD

In [ ]:
# 3. Build the isolated Colab environment and hard-assert an A100
import os
REPO_DIR = '/content/hyperkvasir-multi-cnn-fusion'
os.chdir(REPO_DIR)
required_files = ['pyproject.toml', 'env/requirements-colab.txt', 'configs/training/finetune_wide_focal.yaml']
missing = [path for path in required_files if not os.path.exists(path)]
if missing:
    print('Current working directory:', os.getcwd())
    !git branch --show-current
    !git rev-parse HEAD
    !find . -maxdepth 3 -type f | sort | head -80
    raise FileNotFoundError(f'Missing required repo files: {missing}. Push latest changes to GitHub and rerun cell 2.')
!python -m pip install -q uv
!uv venv --python 3.11 .venv
# `uv pip install -r` resolves transitive deps (typing_extensions, sympy, etc.).
# Do NOT use `uv pip sync` here: sync installs ONLY the listed packages and skips
# transitive deps unless the file is a fully-locked requirements set.
!uv pip install --python .venv/bin/python -r env/requirements-colab.txt
!uv run --no-sync python -c "import torch, torchvision; print('torch', torch.__version__); print('torchvision', torchvision.__version__); print('torch_cuda', torch.version.cuda); print('cuda_available', torch.cuda.is_available()); assert torch.cuda.is_available(), 'CUDA unavailable'; name=torch.cuda.get_device_name(0); print('device', name); assert 'A100' in name, f'A100 required, found {name}'; print(torch.ones(1, device='cuda'))"

In [ ]:
# 3b. (Optional) Freeze the exact working environment as a committable lockfile.
# Run ONLY after cell 3 succeeds. Captures every transitive dep at its installed
# version so the env can be reproduced 1:1 later via:
#   uv pip sync --python .venv/bin/python env/requirements-colab.lock.txt
# Writes to the repo (ephemeral) AND to Drive (persists) — then download the
# Drive copy and commit env/requirements-colab.lock.txt from your local machine.
import os, subprocess
os.chdir('/content/hyperkvasir-multi-cnn-fusion')
drive_root = os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive/hyperkvasir-multi-cnn-fusion')
header = (
    '# Fully-resolved lock from `uv pip freeze` on a working Colab A100.\n'
    '# Reproduce exactly:\n'
    '#   uv pip sync --python .venv/bin/python env/requirements-colab.lock.txt\n'
    '--index-url https://download.pytorch.org/whl/cu128\n'
    '--extra-index-url https://pypi.org/simple\n'
)
frozen = subprocess.check_output(['uv', 'pip', 'freeze', '--python', '.venv/bin/python'], text=True)
lock_text = header + frozen
with open('env/requirements-colab.lock.txt', 'w') as fh:
    fh.write(lock_text)
drive_lock = f'{drive_root}/returned_outputs/requirements-colab.lock.txt'
os.makedirs(os.path.dirname(drive_lock), exist_ok=True)
with open(drive_lock, 'w') as fh:
    fh.write(lock_text)
print(f'[OK] froze {len(frozen.splitlines())} packages')
print('  env/requirements-colab.lock.txt            (ephemeral Colab copy)')
print(f'  {drive_lock}  (download this & commit)')

In [ ]:
# 4. CONTROL PANEL — the ONLY cell you edit per run. All other cells are generic.
import os
os.chdir('/content/hyperkvasir-multi-cnn-fusion')
os.environ['DRIVE_ROOT'] = '/content/drive/MyDrive/hyperkvasir-multi-cnn-fusion'

# --- per-step presets: keep exactly ONE block active ---
# Step 3 (focal 5-fold) — ACTIVE:
os.environ['RUN_ID'] = 'colab_a100_exp16_focal_5fold'
os.environ['EXPERIMENT'] = '16_triple_weighted_finetune_focal_official'
os.environ['TRAINING_CONFIG'] = 'configs/training/finetune_wide_focal.yaml'
os.environ['FOLDS'] = '0 1 2 3 4'

# Step 6 (recipe-v2) — example, leave commented until that step is approved:
# os.environ['RUN_ID'] = 'colab_a100_exp17_recipev2_5fold'
# os.environ['EXPERIMENT'] = '17_triple_weighted_finetune_v2_official'
# os.environ['TRAINING_CONFIG'] = 'configs/training/finetune_wide_v2.yaml'
# os.environ['FOLDS'] = '0 1 2 3 4'

!uv run --no-sync python scripts/write_resolved_config.py --experiment $EXPERIMENT --training $TRAINING_CONFIG --run-id $RUN_ID --device cuda --folds $FOLDS

In [ ]:
# 5. Stage the approved Drive dataset onto local Colab storage (fail early if missing)
import os, shutil
os.chdir('/content/hyperkvasir-multi-cnn-fusion')
src = f"{os.environ['DRIVE_ROOT']}/data/hyperkvasir/labeled-images"
if not os.path.isdir(src):
    raise FileNotFoundError(f'Approved dataset not found on Drive: {src}')
dst = 'data/raw/hyperkvasir/labeled-images'
os.makedirs('data/raw/hyperkvasir', exist_ok=True)
if os.path.isdir(dst):
    print(f'[skip] already staged at {dst}')
else:
    shutil.copytree(src, dst)
n_files = sum(len(files) for _, _, files in os.walk(dst))
print(f'[OK] staged {n_files} files -> {dst}')

In [ ]:
# 6. Run the hard-stop provenance gate before training
import os, subprocess
os.chdir('/content/hyperkvasir-multi-cnn-fusion')
os.environ['EXPECTED_GIT_SHA'] = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
!uv run --no-sync python scripts/check_provenance.py --run-id $RUN_ID --source-dataset-root "$DRIVE_ROOT/data/hyperkvasir/labeled-images" --staged-dataset-root data/raw/hyperkvasir/labeled-images --manifest data/splits/hyperkvasir_official_5fold/fold_0.csv --approved-source "$DRIVE_ROOT/data/hyperkvasir/labeled-images" --expected-git-sha $EXPECTED_GIT_SHA --device cuda

In [ ]:
# 7. Run repository training only. The user executes this GPU cell.
import os
os.chdir('/content/hyperkvasir-multi-cnn-fusion')
!uv run --no-sync python scripts/run_cv.py --experiment $EXPERIMENT --folds $FOLDS --training $TRAINING_CONFIG --device cuda

In [ ]:
# 8. Collect the fixed artifact set back to Drive without overwrite
import os
os.chdir('/content/hyperkvasir-multi-cnn-fusion')
!uv run --no-sync python scripts/collect_outputs.py --run-id $RUN_ID --experiment $EXPERIMENT --folds $FOLDS --destination-root "$DRIVE_ROOT/returned_outputs"

In [ ]:
# 9. Final returned-artifact checklist (hard-fail on any missing file)
import os
RUN_ID = os.environ['RUN_ID']
DRIVE_ROOT = os.environ['DRIVE_ROOT']
base = f'{DRIVE_ROOT}/returned_outputs/{RUN_ID}'
required = [
    f'{base}/resolved_config.yaml',
    f'{base}/runtime.json',
    f'{base}/{RUN_ID}_provenance.json',
]
for f in os.environ['FOLDS'].split():
    required += [f'{base}/fold_{f}/{name}' for name in ('best.pt', 'metrics.json', 'predictions.npz')]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    raise FileNotFoundError('Missing returned artifacts:\n  ' + '\n  '.join(missing))
print(f'[OK] all {len(required)} required artifacts present for {RUN_ID}')